In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
import seaborn as sns
import sys

sys.path.append('../../processors')
from json_parser import flatten_data # type: ignore

pd.set_option('display.max_columns', None)

In [ ]:
json_file = '../../data/raw/AN2023.json' 

df = pd.read_json(json_file)
df = flatten_data(json_file)
# Might also want to extract descrizione_impianto for future use

print(df.shape) # Rows, Columns


In [ ]:
df['edificio_id'] = df.index + 1
df.head()

## Missing values

In [ ]:
# features with missing values
features_na = [features for features in df.columns if df[features].isnull().sum() > 1]

# feature name + percentage of missing values
for feature in features_na:
    print(feature, np.round(df[feature].isnull().mean(), 5), '% missing values')

### Relationship between missing values and the target variable (EPnren)

In [ ]:

for feature in features_na:
    data = df.copy()
    
    # 1 to indicate missing, 0 NOT missing
    data[feature] = np.where(data[feature].isnull(), 1, 0)
    
    # calculate the mean of the target for both groups
    data.groupby(feature)['epglnren'].median().plot.bar()
    plt.title(feature)
    plt.show()

## Numerical values

In [ ]:
numerical_features = df.select_dtypes(include=['number']).columns.tolist()
print('Number of numerical features: ', len(numerical_features))

df[numerical_features].head()

## Temporal variables

In [ ]:
time_features = [feature for feature in df if 'sopralluogo' in feature or 'validita' in feature]
time_features

In [ ]:
# exploring the content of these features

for feature in time_features:
    print(feature, df[feature].unique())  # did not consider anno_costruzione because it is a numerical predictor, not a time feature

### Is there any relation between anno_costruzione and epglnren?

In [ ]:
df.groupby('anno_costruzione')['epglnren'].median().plot.bar(figsize=(12, 6))
plt.xlabel('anno_costruzione')
plt.ylabel('Median epglnren')
plt.title('Median epglnren by anno_costruzione')
plt.xticks(rotation=70, ha='right', fontsize=5)
plt.tight_layout()
plt.show()

##### no further data exploration required for the temporal variables, since they have no relation to the prediction output

## Continuous and Discrete variables

In [ ]:
discrete_features = [feature for feature in df if len(df[feature].unique()) < 5 and feature not in time_features]
print('Discrete features count: ', len(discrete_features))

In [ ]:
discrete_features

In [ ]:
# exploring the relationship between the discrete features and the target variable epglnren
for feature in discrete_features:
    try:
        grouped = df.groupby(feature)['epglnren'].median()
        # remove NaN medians and empty groups
        grouped = grouped.dropna()
        if grouped.empty:
            print(f"Skipping '{feature}': no groups or all NaN medians (unique count: {df[feature].nunique(dropna=True)})")
            continue
        plt.figure(figsize=(6, 4))
        grouped.plot.bar()
        plt.xlabel(feature)
        plt.ylabel('epglnren')
        plt.title(feature)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error plotting feature '{feature}': {e}")
        # diagnostic information
        print('dtype:', df[feature].dtype, 'non-null:', df[feature].notnull().sum(), 'unique:', df[feature].nunique(dropna=True))
        try:
            display(df[[feature, 'epglnren']].head())
        except Exception:
            pass

In [ ]:
# unstructured mixed text features (most likely unwanted and not useful for the prediction task)
unstructured_text_features = ['software_utilizzato', 'cap', 'comune', 'codice_istat_comune', 'informazioni_aggiuntive', 'informazioni_miglioramento', 'piano', 'altra_motivazione', 'provincia', 'comune']
print('Unstructured text features count: ', len(unstructured_text_features))

print('Unstructured text features: ', unstructured_text_features)

In [ ]:
# Continuous features
continuous_features = [feature for feature in df if feature not in discrete_features + time_features + unstructured_text_features and feature != 'classe_energetica' and feature != 'classe_energetica_raggiungibile']
print('Continuous features count: ', len(continuous_features))

In [ ]:
# histograms for continuous features
for feature in continuous_features:
    data = df.copy()
    data[feature].hist(bins=30)
    plt.xlabel(feature)
    plt.ylabel('Count')
    plt.title(feature)
    plt.show()

In [ ]:
## logarithmic transformation for skewed features
for feature in continuous_features:
    data = df.copy()
    data = data[(data[feature] > 0) & (data['epglnren'] > 0)]
    
    if len(data) > 0:
        data[feature] = np.log(data[feature])
        data['epglnren'] = np.log(data['epglnren'])
        plt.scatter(data[feature], data['epglnren'])
        plt.xlabel(feature)
        plt.ylabel('epglnren')
        plt.title(feature)
        plt.show()
    else:
        print(f'{feature} has no positive values, skipping.')

## Outliers

In [ ]:
for feature in continuous_features:
    data = df.copy()
    data = data[(data[feature] > 0) & (data['epglnren'] > 0)]
    
    if len(data) > 0:
        data[feature] = np.log(data[feature])
        data.boxplot(column=feature)
        plt.ylabel(feature)
        plt.title(feature)
        plt.show()
    else:
        print(f'{feature} has no positive values, skipping.')

In [ ]:
## Categorical features
categorical_features = ['classe_energetica', 'classe_energetica_raggiungibile', 'zona_climatica']
categorical_features

In [ ]:
df[categorical_features].head()

In [ ]:
for feature in categorical_features:
    print(f'feature: {feature}, number of categories: {len(df[feature].unique())}')

In [ ]:
## relationship between categorical features and the target variable epglnren
for feature in categorical_features:
    df.groupby(feature)['epglnren'].median().plot.bar()
    plt.xlabel(feature)
    plt.ylabel('epglnren')
    plt.title(feature)
    plt.show()

## Correlation matrix

In [ ]:
corr = df[numerical_features].corr()
plt.figure(figsize=(50, 30))
mask = corr.isnull()
# copy where NaNs are a distinct value just for coloring
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', linewidths=.5,
            cbar_kws={'label': 'Pearson r'})
# overlay text at NaN locations:
for i, j in zip(*np.where(mask)):
    plt.text(j+0.5, i+0.5, 'NA', ha='center', va='center', color='black', fontsize=6)
plt.show()

### Structural issue with the servizio_* features, these features are very sparse and conditional, with semantic inconsistency 

In [ ]:
# The servizio_* features are very sparse and conditional, with semantic inconsistency. We will reshape them into a long format for better analysis.

servizi = []

for i in range(0, 8):  # servizio_0 to servizio_7
    cols = {
        "simulato": f"servizio_{i}_simulato",
        "epnren": f"servizio_{i}_epnren",
        "epren": f"servizio_{i}_epren",
        "efficienza": f"servizio_{i}_efficienza",
        "potenza_nominale": f"servizio_{i}_pnominale"
    }
    
    existing = [c for c in cols.values() if c in df.columns]

    if len(existing) < 2:  # If less than 2 of the expected
        continue           # Skip this servizio index
    
    temp = df[['edificio_id'] + existing].copy()

    rename_map = {
        cols["simulato"]: "simulato",
        cols["epnren"]: "epnren",
        cols["epren"]: "epren",
        cols["efficienza"]: "efficienza",
        cols["potenza_nominale"]: "potenza_nominale"
    }

    temp = temp.rename(columns = {k: v for k, v in rename_map.items() if k in temp.columns})
    temp["servizio_idx"] = i
    
    servizi.append(temp)

long_df = pd.concat(servizi, ignore_index=True)
long_df = long_df.dropna(subset=["epnren", 'efficienza'], how = "all") # Drop rows where the system doesn't exist
counts = long_df['servizio_idx'].value_counts().sort_index()
print(counts)

In [ ]:
long_df.head() # edificio_id is important here

### Extract information and useful consistent features, then drop all raw servizio_* features

In [ ]:
# Features related to efficiency
efficiency_stats = long_df.groupby('edificio_id')['efficienza'].agg(
    efficienza_media = 'mean',
    efficienza_min = 'min',
    efficienza_max = 'max',
    efficienza_std = 'std'
)

# Servizi counts
num_servizi = long_df.groupby('edificio_id').size().rename('num_servizi')

# Presence of servizi
# servizio_presence = pd.crosstab(
#     long_df['servizio_idx'], 
#     long_df['simulato']
# )
# servizio_presence.columns = [f"has_{col}" for col in servizio_presence.columns]

# Real vs Simulato
real_df = long_df[long_df['simulato'] == 0]
sim_df = long_df[long_df['simulato'] == 1]
real_efficienza = real_df.groupby('edificio_id')['efficienza'].mean().rename('efficienza_media_reale')
sim_efficienza = sim_df.groupby('edificio_id')['efficienza'].mean().rename('efficienza_media_simulata')
# Simulation features
sim_stats = long_df.groupby('edificio_id')['simulato'].agg(
    num_simulati = 'sum',
    perc_simulati = 'mean'
)

# Potenza nominale stats
potenza_stats = long_df.groupby('edificio_id')['potenza_nominale'].agg(
    potenza_tot = 'sum',
    potenza_media = 'mean',
    potenza_max = 'max',
    potenza_std = 'std'
)

# Worst servizio
# worst_servizio = long_df.loc[long_df.groupby('simulato')['efficienza'].idxmin()][['simulato', 'servizio_idx']]
# worst_servizio = worst_servizio.rename(columns = {
#     'servizio_idx': 'worst_servizio_idx'
# })

# EPNREN stats (careful not to cause leakage)
long_df['epnren_norm'] = long_df['epnren'] / long_df.groupby('edificio_id')['epnren'].transform('sum')
epnren_stats = long_df.groupby('edificio_id')['epnren_norm'].agg(
    epnren_max_share = 'max',
    epnren_std = 'std'
)

In [ ]:

# Merge everything back together
df_new = df.copy()

for feature in [num_servizi, efficiency_stats, real_efficienza, sim_efficienza, sim_stats, potenza_stats, epnren_stats]: # worst_servizio
    df_new = df_new.merge(feature, on='edificio_id', how='left')

servizio_cols = [c for c in df.columns if c.startswith('servizio_')]
df_new = df_new.drop(columns=servizio_cols)


In [ ]:
df_new.shape

In [ ]:
df_new.head()

## New correlation heatmap for new_df

In [ ]:
numerical_features_new = df_new.select_dtypes(include=['number']).columns.tolist()
print('Number of numerical features: ', len(numerical_features_new))

corr_new = df_new[numerical_features_new].corr()
plt.figure(figsize=(30, 20))
mask = corr_new.isnull()
# copy where NaNs are a distinct value just for coloring
sns.heatmap(corr_new, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', linewidths=.5,
            cbar_kws={'label': 'Pearson r'})
# overlay text at NaN locations:
for i, j in zip(*np.where(mask)):
    plt.text(j+0.5, i+0.5, 'NA', ha='center', va='center', color='black', fontsize=6)
plt.show()